### Loading the California Housing Dataset
In this cell, we fetch the California Housing dataset using the `fetch_california_housing` function from `sklearn.datasets` and convert it into a pandas DataFrame for analysis.

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# California Housing verisini çekelim
california_dataset = fetch_california_housing(as_frame=True)

# DataFrame'i elde edelim
df = california_dataset.frame

# Veri setine ait bilgileri yazdıralım
print("Veri Seti Boyutu:", df.shape)
display(df.head())

### Splitting the Dataset into Train, Validation, and Test Sets
In this step, we split the data into features (`X`) and target labels (`y`: `MedHouseVal`). Following the book's practice, we split them into training (`X_train`, `y_train`), validation (`X_valid`, `y_valid`), and test (`X_test`, `y_test`) sets.

In [2]:
from sklearn.model_selection import train_test_split

# Hedef değişken (MedHouseVal) ve özellikleri ayıralım
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

# İlk olarak %80 train+valid, %20 test olacak şekilde bölüyoruz
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# train+valid setini de kendi içinde %80 train, %20 valid olacak şekilde tekrar bölüyoruz
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

# Boyutları kontrol edelim
print("X_train boyutu:", X_train.shape)
print("X_valid boyutu:", X_valid.shape)
print("X_test boyutu:", X_test.shape)
print("y_train boyutu:", y_train.shape)
print("y_valid boyutu:", y_valid.shape)
print("y_test boyutu:", y_test.shape)

X_train boyutu: (13209, 8)
X_valid boyutu: (3303, 8)
X_test boyutu: (4128, 8)
y_train boyutu: (13209,)
y_valid boyutu: (3303,)
y_test boyutu: (4128,)


In [5]:
import torch

# Convert pandas DataFrames to PyTorch float tensors using .values
X_train = torch.FloatTensor(X_train.values)
X_valid = torch.FloatTensor(X_valid.values)
X_test = torch.FloatTensor(X_test.values)

# Also convert targets to float tensors and reshape them to 2D column vectors as typical for neural networks
y_train = torch.FloatTensor(y_train.values).view(-1, 1)
y_valid = torch.FloatTensor(y_valid.values).view(-1, 1)
y_test = torch.FloatTensor(y_test.values).view(-1, 1)

print("PyTorch Tensors ready:")
print("X_train tensor shape:", X_train.shape)
print("y_train tensor shape:", y_train.shape)

PyTorch Tensors ready:
X_train tensor shape: torch.Size([13209, 8])
y_train tensor shape: torch.Size([13209, 1])


In [11]:
means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdims=True)

torch.Size([1, 8])

In [9]:
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

In [12]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [13]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
  y_pred = X_train @ w + b
  loss = ((y_pred - y_train) ** 2).mean()
  loss.backward()
  with torch.no_grad():
    b -= learning_rate * b.grad
    w -= learning_rate * w.grad
    b.grad.zero_()
    w.grad.zero_()
  print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

Epoch 1/20, Loss: 14.354253768920898
Epoch 2/20, Loss: 4.658701419830322
Epoch 3/20, Loss: 2.145606279373169
Epoch 4/20, Loss: 1.261646032333374
Epoch 5/20, Loss: 0.922481119632721
Epoch 6/20, Loss: 0.7815980911254883
Epoch 7/20, Loss: 0.7158308625221252
Epoch 8/20, Loss: 0.6797963976860046
Epoch 9/20, Loss: 0.6563626527786255
Epoch 10/20, Loss: 0.6388931274414062
Epoch 11/20, Loss: 0.6247153878211975
Epoch 12/20, Loss: 0.6126787662506104
Epoch 13/20, Loss: 0.60222989320755
Epoch 14/20, Loss: 0.5930587649345398
Epoch 15/20, Loss: 0.5849621295928955
Epoch 16/20, Loss: 0.577788770198822
Epoch 17/20, Loss: 0.5714176893234253
Epoch 18/20, Loss: 0.5657473802566528
Epoch 19/20, Loss: 0.5606915950775146
Epoch 20/20, Loss: 0.556175947189331


In [14]:
import torch.nn as nn

In [15]:
torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

In [16]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [17]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [23]:
[par for par in model.parameters()]

[Parameter containing:
 tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
        requires_grad=True),
 Parameter containing:
 tensor([0.3117], requires_grad=True)]

In [25]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()


In [27]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
  for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [28]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.270206928253174
Epoch 2/20, Loss: 0.7620965242385864
Epoch 3/20, Loss: 0.611318051815033
Epoch 4/20, Loss: 0.5919573307037354
Epoch 5/20, Loss: 0.5812456011772156
Epoch 6/20, Loss: 0.5727838277816772
Epoch 7/20, Loss: 0.5656696557998657
Epoch 8/20, Loss: 0.5595564246177673
Epoch 9/20, Loss: 0.5542508363723755
Epoch 10/20, Loss: 0.5496223568916321
Epoch 11/20, Loss: 0.5455719828605652
Epoch 12/20, Loss: 0.5420191884040833
Epoch 13/20, Loss: 0.5388969779014587
Epoch 14/20, Loss: 0.5361483097076416
Epoch 15/20, Loss: 0.5337242484092712
Epoch 16/20, Loss: 0.5315828323364258
Epoch 17/20, Loss: 0.5296877026557922
Epoch 18/20, Loss: 0.5280078053474426
Epoch 19/20, Loss: 0.5265161395072937
Epoch 20/20, Loss: 0.525189220905304
